
## Folder 10 — Logging, Checkpoints and Recovery:


This notebook manages and validates the checkpoint directories used by your E-Commerce Lakehouse Auto Loader pipelines.

It covers:

- Checkpoint path configuration.
- Checkpoint directory existence.
- Checkpoint file inspection.
- Checkpoint health reporting.
- Safe backup procedures.
- Recovery readiness validation.


**Important:**

- A checkpoint stores processing state. 
- Deleting it can cause previously processed files to be ingested again. 
- This notebook will not automatically delete or reset checkpoints.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/"
    f"{VOLUME}/ecommerce_lakehouse"
)

CHECKPOINT_BASE_PATH = f"{BASE_PATH}/checkpoints"

BACKUP_BASE_PATH = f"{BASE_PATH}/checkpoint_backups"

LOG_TABLE = (
    "workspace.ecommerce_gold."
    "checkpoint_management_report"
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_gold"
)


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

DATASETS = [

    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"

]


# ============================================================
# 4. DEFINE CHECKPOINT LOCATIONS
# ============================================================

def get_checkpoint_path(dataset_name):

    return (
        f"{CHECKPOINT_BASE_PATH}/bronze/"
        f"{dataset_name}"
    )


def get_schema_path(dataset_name):

    return (
        f"{BASE_PATH}/schemas/"
        f"bronze/{dataset_name}"
    )


# ============================================================
# 5. CHECKPOINT DIRECTORY INSPECTION
# ============================================================

def inspect_checkpoint(path):

    result = {

        "path": path,

        "exists": False,

        "file_count": 0,

        "directory_count": 0,

        "total_size_bytes": 0,

        "inspection_status": "NOT_FOUND",

        "error_message": None

    }

    try:

        items = dbutils.fs.ls(path)

        result["exists"] = True

        result["inspection_status"] = "EXISTS"

        for item in items:

            if item.isDir():

                result["directory_count"] += 1

            else:

                result["file_count"] += 1

                result["total_size_bytes"] += item.size

    except Exception as error:

        result["error_message"] = str(error)[:2000]

    return result


# ============================================================
# 6. CHECKPOINT VALIDATION
# ============================================================

def validate_checkpoint(dataset_name):

    checkpoint_path = get_checkpoint_path(
        dataset_name
    )

    schema_path = get_schema_path(
        dataset_name
    )

    checkpoint_info = inspect_checkpoint(
        checkpoint_path
    )

    schema_info = inspect_checkpoint(
        schema_path
    )

    if not checkpoint_info["exists"]:

        status = "MISSING_CHECKPOINT"

    elif not schema_info["exists"]:

        status = "MISSING_SCHEMA_DIRECTORY"

    else:

        status = "EXISTS"

    return {

        "dataset_name": dataset_name,

        "checkpoint_path": checkpoint_path,

        "schema_path": schema_path,

        "checkpoint_exists": checkpoint_info["exists"],

        "checkpoint_file_count": checkpoint_info["file_count"],

        "checkpoint_directory_count": checkpoint_info["directory_count"],

        "checkpoint_size_bytes": checkpoint_info["total_size_bytes"],

        "schema_exists": schema_info["exists"],

        "schema_file_count": schema_info["file_count"],

        "validation_status": status,

        "error_message": (
            checkpoint_info["error_message"]
            or schema_info["error_message"]
        )

    }


# ============================================================
# 7. RUN CHECKPOINT VALIDATION
# ============================================================

checkpoint_results = []

for dataset in DATASETS:

    result = validate_checkpoint(dataset)

    checkpoint_results.append(result)

    print(
        f"{dataset}: {result['validation_status']}"
    )


# ============================================================
# 8. CREATE CHECKPOINT REPORT DATAFRAME
# ============================================================

report_schema = StructType([

    StructField("run_id", StringType(), False),

    StructField("dataset_name", StringType(), True),

    StructField("checkpoint_path", StringType(), True),

    StructField("schema_path", StringType(), True),

    StructField("checkpoint_exists", BooleanType(), True),

    StructField("checkpoint_file_count", IntegerType(), True),

    StructField("checkpoint_directory_count", IntegerType(), True),

    StructField("checkpoint_size_bytes", LongType(), True),

    StructField("schema_exists", BooleanType(), True),

    StructField("schema_file_count", IntegerType(), True),

    StructField("validation_status", StringType(), True),

    StructField("error_message", StringType(), True),

    StructField("checked_at", TimestampType(), True)

])


RUN_ID = str(uuid.uuid4())

CHECKED_AT = datetime.now(timezone.utc).replace(
    tzinfo=None
)

report_records = []

for result in checkpoint_results:

    report_records.append({

        "run_id": RUN_ID,

        **result,

        "checked_at": CHECKED_AT

    })


checkpoint_report_df = spark.createDataFrame(

    report_records,

    schema=report_schema

)


# ============================================================
# 9. SAVE CHECKPOINT VALIDATION REPORT
# ============================================================

(

    checkpoint_report_df

    .write

    .format("delta")

    .mode("append")

    .saveAsTable(LOG_TABLE)

)

print(
    f"Checkpoint report saved: {LOG_TABLE}"
)


# ============================================================
# 10. DISPLAY CHECKPOINT VALIDATION
# ============================================================

display(

    checkpoint_report_df.orderBy(
        "dataset_name"
    )

)


# ============================================================
# 11. CHECKPOINT HEALTH SUMMARY
# ============================================================

checkpoint_summary = (

    checkpoint_report_df

    .groupBy("validation_status")

    .agg(

        F.count("*").alias("dataset_count")

    )

    .orderBy("validation_status")

)

display(checkpoint_summary)


# ============================================================
# 12. CREATE BACKUP DIRECTORY
# ============================================================

dbutils.fs.mkdirs(BACKUP_BASE_PATH)

print(
    f"Backup root directory: {BACKUP_BASE_PATH}"
)


# ============================================================
# 13. SAFE CHECKPOINT BACKUP FUNCTION
# ============================================================

def backup_checkpoint(dataset_name, confirm_stopped=False):

    if not confirm_stopped:

        raise ValueError(
            "Stop the associated streaming job first. "
            "Set confirm_stopped=True only after "
            "confirming that no process is writing "
            "to this checkpoint."
        )

    source_path = get_checkpoint_path(
        dataset_name
    )

    source_info = inspect_checkpoint(
        source_path
    )

    if not source_info["exists"]:

        raise FileNotFoundError(
            f"Checkpoint does not exist: {source_path}"
        )

    backup_run_id = str(uuid.uuid4())

    backup_path = (
        f"{BACKUP_BASE_PATH}/"
        f"{dataset_name}/{backup_run_id}"
    )

    dbutils.fs.mkdirs(
        f"{BACKUP_BASE_PATH}/{dataset_name}"
    )

    dbutils.fs.cp(
        source_path,
        backup_path,
        True
    )

    backup_info = inspect_checkpoint(
        backup_path
    )

    if not backup_info["exists"]:

        raise RuntimeError(
            "Checkpoint backup could not be verified."
        )

    return {

        "dataset_name": dataset_name,

        "source_path": source_path,

        "backup_path": backup_path,

        "backup_run_id": backup_run_id,

        "backup_file_count": backup_info["file_count"],

        "backup_status": "BACKUP_CREATED"

    }


# ============================================================
# 14. BACKUP EXAMPLE
# ============================================================

# IMPORTANT:
# Do not execute until the relevant streaming job is stopped.
#
# Example:
#
# backup_result = backup_checkpoint(
#     dataset_name="customers",
#     confirm_stopped=True
# )
#
# print(backup_result)
#
# The backup path is separate from the active checkpoint.
# This operation does not reset or modify the source checkpoint.


# ============================================================
# 15. CHECKPOINT RECOVERY READINESS
# ============================================================

def check_recovery_readiness(dataset_name):

    checkpoint_path = get_checkpoint_path(
        dataset_name
    )

    backup_root = (
        f"{BACKUP_BASE_PATH}/{dataset_name}"
    )

    active_info = inspect_checkpoint(
        checkpoint_path
    )

    backup_info = inspect_checkpoint(
        backup_root
    )

    if not active_info["exists"]:

        status = "ACTIVE_CHECKPOINT_MISSING"

    elif not backup_info["exists"]:

        status = "ACTIVE_EXISTS_BACKUP_NOT_FOUND"

    else:

        status = "ACTIVE_AND_BACKUP_DIRECTORIES_EXIST"

    return {

        "dataset_name": dataset_name,

        "active_checkpoint_exists": active_info["exists"],

        "backup_directory_exists": backup_info["exists"],

        "recovery_readiness": status

    }


recovery_results = [

    check_recovery_readiness(dataset)

    for dataset in DATASETS

]


display(
    spark.createDataFrame(recovery_results)
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

total_datasets = len(checkpoint_results)

existing_checkpoints = sum(

    1 for result in checkpoint_results

    if result["checkpoint_exists"]

)

missing_checkpoints = (
    total_datasets - existing_checkpoints
)

print("\n" + "=" * 65)

print("CHECKPOINT MANAGEMENT SUMMARY")

print("=" * 65)

print(f"Total datasets: {total_datasets}")

print(f"Existing checkpoints: {existing_checkpoints}")

print(f"Missing checkpoints: {missing_checkpoints}")

print(f"Run ID: {RUN_ID}")

print(f"Report table: {LOG_TABLE}")

print("=" * 65)

print("CHECKPOINT INSPECTION COMPLETED.")


In [0]:
%sql
-- Latest checkpoint validation results

SELECT
    dataset_name,
    checkpoint_exists,
    checkpoint_file_count,
    schema_exists,
    validation_status,
    checked_at
FROM workspace.ecommerce_gold.checkpoint_management_report
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY dataset_name
    ORDER BY checked_at DESC
) = 1
ORDER BY dataset_name;

In [0]:
%sql
SELECT
    run_id,
    dataset_name,
    checkpoint_exists,
    checkpoint_file_count,
    checkpoint_directory_count,
    checkpoint_size_bytes,
    schema_exists,
    validation_status,
    checked_at
FROM workspace.ecommerce_gold.checkpoint_management_report
ORDER BY checked_at DESC, dataset_name;